# CancerFusion AI Phase 2 | Breast cancer survival research

This notebook audits the source outcomes and modality availability, trains the four-branch fusion model when the fixed split has observed events, and evaluates relative risk on observed follow-up. It exports per-case predictions, confidence intervals and paired modality ablations. The archived pilot checkpoint and metrics are unchanged.


In [ ]:
!pip install -q lifelines pydicom
import os, glob, json, math
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights, resnet18, ResNet18_Weights
from lifelines import KaplanMeierFitter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Device: {DEVICE}")

In [ ]:
CFG = {
    'dataset': 'breast-cancer-vision-and-genomic-fusion-ml-ready',
    'dir_multimodal': '/kaggle/input/breast-cancer-vision-and-genomic-fusion-ml-ready',
    'batch_size': 32,
    'epochs': 50,
    'lr': 1e-3,
    'max_patches': 20, # Optimized to fit 4 modalities smoothly in GPU RAM
    'num_workers': 0,
    'cache_frozen_features': True,  # CPU RAM; recreated when loaders are rebuilt
    'seed': 42,
    'bootstrap_replicates': 1000,
    # Set only after verifying which numeric predictors are known at prediction time.
    'clinical_baseline_columns': [],
    'image_manifest_csv': '',  # REQUIRED: reviewed patient/modality/path identity CSV.
    'clinical_feature_columns': [],  # REQUIRED: reviewed diagnosis-time predictors; never follow-up.
    'clinical_csv': 'Clinical_Treatment_Data.csv',
    'demographics_csv': 'Clinical_Demographic_Data.csv',
    'omics_csv': 'RNA_CNV_ModelReady.csv',
}

import random

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])

# Auto-Locate Dataset Path in Kaggle (handles both direct and nested mounts)
possible_paths = [
    CFG['dir_multimodal'],
    '/kaggle/input/datasets/sepehreslamimoghadam/breast-cancer-vision-and-genomic-fusion-ml-ready',
    '/kaggle/input/sepehreslamimoghadam/breast-cancer-vision-and-genomic-fusion-ml-ready',
] + glob.glob('/kaggle/input/*breast-cancer*') + glob.glob('/kaggle/input/*/*breast-cancer*')

for p in possible_paths:
    if os.path.exists(p):
        CFG['dir_multimodal'] = p
        break

print(f"[*] Multimodal Dataset Root Auto-Detected At: {CFG['dir_multimodal']}")


In [ ]:
print("[*] Loading WSI Backbone (ViT-B/16)...")
wsi_model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
wsi_model.heads = nn.Identity()
wsi_model = wsi_model.to(DEVICE).eval()

print("[*] Loading Radiology Backbone (ResNet18)...")
rad_model = resnet18(weights=ResNet18_Weights.DEFAULT)
rad_model.fc = nn.Identity()
rad_model = rad_model.to(DEVICE).eval()

img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
"""Validated manifest selection and geometric ordering for research image inputs.

Raster identity must come from a reviewed manifest, never from its extension.
DICOM sequences must be selected explicitly before ordering; this code does not
choose clinically appropriate MRI protocols or establish prediction-time eligibility.
"""

import csv
from pathlib import Path
import numpy as np


def load_image_manifest(manifest_path, image_root):
    root = Path(image_root).resolve()
    groups = {}
    seen = set()
    with Path(manifest_path).open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if not {"patient_id", "modality", "relative_path", "identity_source"} <= set(
            reader.fieldnames or []
        ):
            raise ValueError(
                "Image manifest requires patient_id, modality, relative_path, identity_source."
            )
        for row in reader:
            patient, modality = (
                row["patient_id"].strip(),
                row["modality"].strip().lower(),
            )
            if not patient or any(token in patient for token in ("/", "\\", "..")):
                raise ValueError("Invalid manifest patient identifier.")
            if modality not in ("wsi", "rad") or not row["identity_source"].strip():
                raise ValueError(
                    "Each image needs a reviewed modality and identity source."
                )
            relative = Path(row["relative_path"])
            path = (root / relative).resolve()
            if (
                relative.is_absolute()
                or not path.is_relative_to(root)
                or not path.is_file()
            ):
                raise ValueError("Manifest image must exist inside the image root.")
            if path in seen:
                raise ValueError(
                    "An image cannot be assigned twice or to multiple patients/modalities."
                )
            if modality == "rad":
                import pydicom

                header = pydicom.dcmread(str(path), stop_before_pixels=True)
                if str(header.get("PatientID", "")).strip() != patient:
                    raise ValueError(
                        "Radiology PatientID must match the reviewed case mapping."
                    )
            seen.add(path)
            groups.setdefault((patient, modality), []).append(path)
    return groups


def order_dicom_series(paths, modality):
    """Check one selected MR/CT series and sort by physical slice position."""
    if modality not in ("MR", "CT"):
        raise ValueError("Only reviewed MR or CT volumes are supported.")
    import pydicom

    if not paths:
        return []
    records = []
    for path in paths:
        ds = pydicom.dcmread(str(path), stop_before_pixels=True)
        if ds.get("Modality") != modality or int(ds.get("NumberOfFrames", 1)) != 1:
            raise ValueError(
                "Expected single-frame matching-modality DICOM; enhanced/multiframe needs a dedicated loader."
            )
        for name in (
            "PatientID",
            "SeriesInstanceUID",
            "FrameOfReferenceUID",
            "ImageOrientationPatient",
            "ImagePositionPatient",
            "Rows",
            "Columns",
            "PixelSpacing",
        ):
            if name not in ds:
                raise ValueError(f"DICOM geometry is missing {name}.")
        for name in ("PatientID", "SeriesInstanceUID", "FrameOfReferenceUID"):
            if not str(ds.get(name, "")).strip():
                raise ValueError(f"DICOM identity is empty: {name}.")
        if int(ds.Rows) <= 0 or int(ds.Columns) <= 0:
            raise ValueError("DICOM Rows and Columns must be positive.")
        orientation = np.asarray(ds.ImageOrientationPatient, dtype=float)
        position = np.asarray(ds.ImagePositionPatient, dtype=float)
        spacing = np.asarray(ds.PixelSpacing, dtype=float)
        if (
            orientation.shape != (6,)
            or position.shape != (3,)
            or spacing.shape != (2,)
            or not np.isfinite(np.r_[orientation, position, spacing]).all()
            or (spacing <= 0).any()
        ):
            raise ValueError("Invalid DICOM geometry.")
        if (
            not np.isclose(np.linalg.norm(orientation[:3]), 1, atol=1e-4)
            or not np.isclose(np.linalg.norm(orientation[3:]), 1, atol=1e-4)
            or not np.isclose(np.dot(orientation[:3], orientation[3:]), 0, atol=1e-4)
        ):
            raise ValueError(
                "DICOM orientation must contain orthonormal direction cosines."
            )
        records.append((Path(path), ds, orientation, spacing, position))
    first = records[0]
    reference_normal = np.cross(first[2][:3], first[2][3:])
    reference_normal /= np.linalg.norm(reference_normal)
    for record in records[1:]:
        if (
            str(record[1].PatientID) != str(first[1].PatientID)
            or str(record[1].SeriesInstanceUID) != str(first[1].SeriesInstanceUID)
            or str(record[1].FrameOfReferenceUID) != str(first[1].FrameOfReferenceUID)
        ):
            raise ValueError("Select one DICOM series and frame of reference per bag.")
        if (
            (int(record[1].Rows), int(record[1].Columns))
            != (int(first[1].Rows), int(first[1].Columns))
            or not np.allclose(record[2], first[2], atol=1e-4)
            or not np.allclose(record[3], first[3], atol=1e-4)
        ):
            raise ValueError("Mixed DICOM orientation, dimensions or pixel spacing.")
        delta = record[4] - first[4]
        in_plane_shift = delta - float(delta @ reference_normal) * reference_normal
        if np.linalg.norm(in_plane_shift) > 1e-3:
            raise ValueError(
                "DICOM slice origins shift within the image plane; resample explicitly before volume feature extraction."
            )
    records.sort(key=lambda record: float(record[4] @ reference_normal))
    gaps = np.diff([float(record[4] @ reference_normal) for record in records])
    if (gaps <= 1e-4).any():
        raise ValueError(
            "Duplicate or overlapping slice positions; select a single acquisition."
        )
    if len(gaps) > 1 and not np.allclose(gaps, np.median(gaps), rtol=0.05, atol=0.1):
        raise ValueError(
            "Nonuniform slice spacing; resample explicitly before feature extraction."
        )
    return [record[0] for record in records]


def select_evenly_spaced(paths, limit):
    if limit < 1:
        raise ValueError("Image item limit must be positive.")
    if len(paths) <= limit:
        return list(paths)
    return [paths[index] for index in np.linspace(0, len(paths) - 1, limit, dtype=int)]


def order_mr_series(paths):
    """Validate selected MR data without inferring sequence identity."""
    return order_dicom_series(paths, "MR")


def order_ct_series(paths):
    """Validate selected CT data; intensity conversion is a separate contract."""
    return order_dicom_series(paths, "CT")


"""Audit the Phase 2 source tables and the image loader's actual file selection."""

from collections import Counter
import json
from pathlib import Path

import numpy as np
import pandas as pd


def audit_outcomes(demographics, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    columns = {
        "status": "demographic_vital_status",
        "death": "demographic_days_to_death",
        "followup": "diagnoses_days_to_last_follow_up",
    }
    missing = [column for column in columns.values() if column not in demographics]
    summary = {"source_rows": len(demographics), "required_columns_missing": missing}
    status_column = columns["status"]
    if status_column in demographics:
        summary["raw_status_counts"] = {
            str(key): int(value)
            for key, value in demographics[status_column].value_counts(dropna=False).items()
        }
    if not missing:
        rows = demographics[["Patient_ID", *columns.values()]].copy()
        status = rows[status_column].astype("string").str.strip().str.casefold()
        rows["event"] = status.map({"alive": 0, "living": 0, "dead": 1, "deceased": 1})
        death = pd.to_numeric(rows[columns["death"]], errors="coerce")
        followup = pd.to_numeric(rows[columns["followup"]], errors="coerce")
        rows["time"] = death.where(rows["event"].eq(1), followup)
        rows["usable"] = rows["event"].notna() & np.isfinite(rows["time"]) & rows["time"].gt(0)
        rows["unmapped_status"] = rows["event"].isna()
        rows["invalid_duration"] = ~np.isfinite(rows["time"]) | rows["time"].le(0)
        rows.to_csv(output_dir / "outcome_rows.csv", index=False)
        summary.update({
            "usable_rows": int(rows["usable"].sum()),
            "usable_event_rows": int((rows["usable"] & rows["event"].eq(1)).sum()),
            "unmapped_status_rows": int(rows["unmapped_status"].sum()),
            "invalid_duration_rows": int(rows["invalid_duration"].sum()),
            "duplicate_patient_rows": int(rows["Patient_ID"].duplicated().sum()),
        })
    (output_dir / "outcome_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


def audit_modality_files(cohort, image_root, rna_ids, clinical_ids, max_items, output_dir, image_manifest=None):
    """Replay the current extension-based routing and decoding without neural inference.

    The manifest describes loader behavior. It does not validate imaging modality identity.
    """
    from PIL import Image
    import pydicom

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    rna_ids, clinical_ids = set(rna_ids), set(clinical_ids)
    cases, files = [], []
    for record in cohort.to_dict("records"):
        patient = str(record["Patient_ID"])
        # Patient identifiers are directory names, never paths supplied to the file system.
        if Path(patient).name != patient or patient in (".", "..") or "/" in patient or "\\" in patient:
            raise ValueError(f"Invalid patient directory identifier: {patient!r}")
        candidates = sorted(path for path in (Path(image_root) / patient).rglob("*") if path.is_file())
        if image_manifest is None:
            # Historical loader audit retained to explain archived runs only.
            wsi = [path for path in candidates if path.suffix in (".jpg", ".png")]
            rad = [path for path in candidates if path.suffix not in (".jpg", ".png")]
        else:
            wsi = sorted(image_manifest.get((patient, 'wsi'), []))
            rad = order_mr_series(image_manifest.get((patient, 'rad'), []))
            wsi, rad = select_evenly_spaced(wsi, max_items), select_evenly_spaced(rad, max_items)
        counts = Counter()
        for modality, paths in (("wsi", wsi), ("rad", rad)):
            for path in paths[:max_items]:
                error = ""
                try:
                    if modality == "wsi":
                        with Image.open(path) as im:
                            im.convert("RGB").resize((224, 224)).load()
                    else:
                        array = pydicom.dcmread(str(path), force=True).pixel_array.astype(float)
                        array = (array - np.min(array)) / (np.max(array) - np.min(array) + 1e-8)
                        array = (array * 255).astype(np.uint8)
                        Image.fromarray(array if array.ndim == 2 else array[0]).convert("RGB").resize((224, 224)).load()
                    counts[modality] += 1
                except Exception as exc:
                    error = f"{type(exc).__name__}: {exc}"
                files.append({
                    "Patient_ID": patient, "loader_branch": modality,
                    "relative_path": path.relative_to(image_root).as_posix(),
                    "decoded": not error, "error": error,
                })
        cases.append({
            **record, "wsi_candidates": len(wsi), "rad_candidates": len(rad),
            "wsi_decoded": counts["wsi"], "rad_decoded": counts["rad"],
            "rna_available": patient in rna_ids, "cli_available": patient in clinical_ids,
            "raster_in_radiology_path": sum(
                any(word in path.as_posix().lower() for word in ("mri", "radiology", "dicom")) for path in wsi
            ),
        })
    frame = pd.DataFrame(cases)
    frame.to_csv(output_dir / "modality_manifest.csv", index=False)
    pd.DataFrame(files, columns=["Patient_ID", "loader_branch", "relative_path", "decoded", "error"]).to_csv(
        output_dir / "image_decode_audit.csv", index=False
    )
    partitions = {}
    for split, rows in frame.groupby("split"):
        partitions[split] = {
            "cases": len(rows), "events": int(rows["event"].sum()),
            "wsi_usable_cases": int(rows["wsi_decoded"].gt(0).sum()),
            "rad_usable_cases": int(rows["rad_decoded"].gt(0).sum()),
            "rna_available_cases": int(rows["rna_available"].sum()),
            "cli_available_cases": int(rows["cli_available"].sum()),
            "all_four_available_cases": int((rows["wsi_decoded"].gt(0) & rows["rad_decoded"].gt(0)
                                              & rows["rna_available"] & rows["cli_available"]).sum()),
        }
    summary = {
        "image_root": str(image_root), "max_items_tested_per_branch": max_items,
        "routing": "Reviewed modality manifest; geometric MR series ordering." if image_manifest is not None else "Historical loader assigns lowercase jpg/png to WSI and other files to RAD.",
        "interpretation": "Decode success does not establish true modality identity or feature quality.",
        "partitions": partitions,
    }
    (output_dir / "modality_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


class PanCancerDataset(Dataset):
    def __init__(self, df, rna_df, cli_df):
        self.df = df
        self.rna_df = rna_df
        self.cli_df = cli_df
        self.feature_cache = {}
        if not CFG.get('image_manifest_csv'):
            raise ValueError('Set image_manifest_csv to a reviewed manifest; extension-based image identity is disabled.')
        self.img_folder = CFG['dir_multimodal']
        self.image_manifest = load_image_manifest(CFG['image_manifest_csv'], self.img_folder)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if not CFG['cache_frozen_features']:
            return self._encode_case(idx)
        if idx not in self.feature_cache:
            self.feature_cache[idx] = self._encode_case(idx)
        # Default collation stacks these CPU tensors into separate batch buffers.
        return self.feature_cache[idx]

    def _encode_case(self, idx):
        patient_id = self.df.iloc[idx]['Patient_ID']
        
        # Initialize Tensors (1 WSI, 2 RAD, 3 RNA, 4 CLI)
        wsi_tensor = torch.zeros((CFG['max_patches'], 768))
        rad_tensor = torch.zeros((CFG['max_patches'], 512))
        wsi_valid = torch.zeros(CFG['max_patches'], dtype=torch.bool)
        rad_valid = torch.zeros(CFG['max_patches'], dtype=torch.bool)
        mask = torch.zeros(4) # [WSI, RAD, RNA, CLI]
        
        # 1. Image Modalities (WSI & Radiology)
        wsi_files = select_evenly_spaced(sorted(self.image_manifest.get((str(patient_id), 'wsi'), [])), CFG['max_patches'])
        rad_files = select_evenly_spaced(order_mr_series(self.image_manifest.get((str(patient_id), 'rad'), [])), CFG['max_patches'])

        # Process WSI
        if len(wsi_files) > 0:
            loaded = []
            for p in wsi_files[:CFG['max_patches']]:
                try:
                    with Image.open(p) as image:
                        loaded.append(img_transform(image.convert('RGB')))
                except: pass
            if loaded:
                mask[0] = 1.0
                wsi_valid[:len(loaded)] = True
                with torch.no_grad():
                    wsi_tensor[:len(loaded)] = wsi_model(torch.stack(loaded).to(DEVICE)).cpu()
                    
        # Process Radiology (DICOM)
        if len(rad_files) > 0:
            loaded = []
            for p in rad_files[:CFG['max_patches']]:
                try:
                    ds = pydicom.dcmread(p, stop_before_pixels=False, force=True)
                    arr = ds.pixel_array.astype(float)
                    arr = (arr - np.min(arr)) / (np.max(arr) - np.min(arr) + 1e-8)
                    arr = (arr * 255).astype(np.uint8)
                    img = Image.fromarray(arr if len(arr.shape)==2 else arr[0]).convert('RGB')
                    loaded.append(img_transform(img))
                except: pass
            if loaded:
                mask[1] = 1.0
                rad_valid[:len(loaded)] = True
                with torch.no_grad():
                    rad_tensor[:len(loaded)] = rad_model(torch.stack(loaded).to(DEVICE)).cpu()

        # 2 & 3. RNA & Clinical
        rna = torch.tensor(self.rna_df.loc[patient_id].values, dtype=torch.float32) if patient_id in self.rna_df.index else torch.zeros(self.rna_df.shape[1])
        cli = torch.tensor(self.cli_df.loc[patient_id].values, dtype=torch.float32) if patient_id in self.cli_df.index else torch.zeros(self.cli_df.shape[1])
        if patient_id in self.rna_df.index: mask[2] = 1.0
        if patient_id in self.cli_df.index: mask[3] = 1.0
        
        # Target
        time = self.df.iloc[idx]['time']
        event = self.df.iloc[idx]['event']
        
        return {'patient_id': str(patient_id), 'wsi': wsi_tensor, 'rad': rad_tensor, 'rna': rna, 'cli': cli, 'mask': mask, 'wsi_valid': wsi_valid, 'rad_valid': rad_valid, 'time': time, 'event': event}

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def _item_mask(x, valid_items=None):
    """Validate bag membership; infer legacy zero padding when unspecified."""
    if x.ndim != 3 or x.shape[1] == 0:
        raise ValueError("Image features require nonempty [batch, items, features].")
    valid = x.ne(0).any(dim=-1) if valid_items is None else valid_items
    if valid.shape != x.shape[:2] or not torch.all((valid == 0) | (valid == 1)):
        raise ValueError("Item mask must be binary with shape [batch, items].")
    valid = valid.bool()
    if not torch.isfinite(x.masked_select(valid.unsqueeze(-1).expand_as(x))).all():
        raise ValueError("Available image features must be finite.")
    return valid


def _safe_attention_mask(valid):
    safe = valid.clone()
    safe[~safe.any(dim=1), 0] = True
    return safe


# =====================================================================
# 🔬 CANCERFUSION RESEARCH PLATFORM: TIER-1 MULTIMODAL ONCOLOGY ARCHITECTURE
# =====================================================================


class GatedAttentionMIL(nn.Module):
    """
    Gated Attention Multiple Instance Learning (Ilse et al., ICML 2018 / CLAM).
    Learns patch pooling weights; tissue semantics require annotation-based validation.
    """

    def __init__(self, in_features=768, hidden_dim=256, out_dim=64, num_concepts=16):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_features, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU()
        )
        self.attention_V = nn.Sequential(nn.Linear(hidden_dim, 128), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(hidden_dim, 128), nn.Sigmoid())
        self.attention_weights = nn.Linear(128, 1)

        # Unsupervised latent bottleneck (no annotated cellular semantics)
        self.concept_layer = nn.Linear(hidden_dim, num_concepts)
        self.concept_proj = nn.Linear(num_concepts, out_dim)
        self.out_norm = nn.LayerNorm(out_dim)

    def forward(self, x, valid_items=None):
        valid = _item_mask(x, valid_items)
        x = x.masked_fill(~valid.unsqueeze(-1), 0)
        # x: [B, N_patches, 768]
        B, N, D = x.shape
        h = self.proj(x)  # [B, N, 256]

        # Gated Attention Scoring: A = softmax(w^T * (tanh(Vh) * sigm(Uh)))
        A_V = self.attention_V(h)  # [B, N, 128]
        A_U = self.attention_U(h)  # [B, N, 128]
        raw_scores = self.attention_weights(A_V * A_U).squeeze(-1)  # [B, N]

        attn_weights = F.softmax(
            raw_scores.masked_fill(~_safe_attention_mask(valid), float("-inf")), dim=1
        )
        attn_weights = attn_weights.masked_fill(~valid, 0)

        # Attentive Bag Pooling
        bag_repr = torch.bmm(attn_weights.unsqueeze(1), h).squeeze(1)  # [B, 256]

        # Concept Bottleneck Activation
        concepts = torch.sigmoid(self.concept_layer(bag_repr))  # [B, num_concepts]
        out = self.out_norm(self.concept_proj(concepts))  # [B, out_dim]

        out = out.masked_fill(~valid.any(dim=1, keepdim=True), 0)
        return out, concepts, attn_weights


class VolumetricSliceTransformer(nn.Module):
    """
    3D-Aware Positional Slice Transformer for Radiology (MRI DICOM Volumes).
    Uses sequence positions; physical anatomical ordering must be established in preprocessing.
    """

    def __init__(
        self, in_features=512, hidden_dim=256, out_dim=64, max_slices=32, nheads=4
    ):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_features, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU()
        )
        self.pos_embedding = nn.Parameter(torch.randn(1, max_slices, hidden_dim) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nheads,
            dim_feedforward=512,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.slice_attention = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.Tanh(), nn.Linear(64, 1)
        )
        self.out_proj = nn.Sequential(
            nn.Linear(hidden_dim, out_dim), nn.LayerNorm(out_dim)
        )

    def forward(self, x, valid_items=None):
        valid = _item_mask(x, valid_items)
        if x.shape[1] > self.pos_embedding.shape[1]:
            raise ValueError("Slice count exceeds positional embedding length.")
        x = x.masked_fill(~valid.unsqueeze(-1), 0)
        # x: [B, S_slices, 512]
        B, S, D = x.shape
        h = self.proj(x)  # [B, S, 256]

        # Add 3D anatomical depth positional embeddings
        h = h + self.pos_embedding[:, :S, :]

        # Cross-slice context encoding
        h_trans = self.transformer(h, src_key_padding_mask=~_safe_attention_mask(valid))

        # Attentive slice pooling
        slice_scores = self.slice_attention(h_trans).squeeze(-1)  # [B, S]
        slice_weights = F.softmax(
            slice_scores.masked_fill(~_safe_attention_mask(valid), float("-inf")), dim=1
        )
        slice_weights = slice_weights.masked_fill(~valid, 0)

        vol_repr = torch.bmm(slice_weights.unsqueeze(1), h_trans).squeeze(1)  # [B, 256]
        out = self.out_proj(vol_repr)  # [B, out_dim]

        out = out.masked_fill(~valid.any(dim=1, keepdim=True), 0)
        return out, slice_weights


class PathwayGenomicSNN(nn.Module):
    """
    Self-Normalizing Neural Network (SNN) with SELU and AlphaDropout for High-Dimensional RNA-Seq.
    Uses an unsupervised latent bottleneck, not annotated biological pathways.
    """

    def __init__(self, in_features=1000, hidden_dim=256, out_dim=64, num_pathways=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.SELU(),
            nn.AlphaDropout(p=0.1),
            nn.Linear(512, hidden_dim),
            nn.SELU(),
        )
        # Unsupervised latent layer (no pathway supervision)
        self.pathway_layer = nn.Linear(hidden_dim, num_pathways)
        self.out_proj = nn.Sequential(
            nn.Linear(num_pathways, out_dim), nn.LayerNorm(out_dim)
        )

    def forward(self, x):
        # x: [B, D_rna]
        h = self.encoder(x)
        pathway_activations = torch.sigmoid(self.pathway_layer(h))  # [B, num_pathways]
        out = self.out_proj(pathway_activations)  # [B, out_dim]
        return out, pathway_activations


class ClinicalDenseEncoder(nn.Module):
    """
    LayerNorm Tabular Encoder with Residual Feed-Forward Blocks for Clinical/Demographic Data.
    """

    def __init__(self, in_features=15, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, out_dim),
            nn.LayerNorm(out_dim),
        )

    def forward(self, x):
        return self.net(x)


class MultiModalCoAttentionTransformer(nn.Module):
    """
    Multi-head fusion with observed availability masks.
    Masking alone does not establish robustness to informative clinical missingness.
    """

    def __init__(self, embed_dim=64, num_modalities=4, nheads=4):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_modalities = num_modalities

        # Learnable Modality Type Embeddings
        self.modality_embeddings = nn.Parameter(
            torch.randn(1, num_modalities, embed_dim) * 0.02
        )

        # Multi-Head Co-Attention
        self.mha = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=nheads, dropout=0.1, batch_first=True
        )
        self.norm1 = nn.LayerNorm(embed_dim)

        # Feed-Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim),
        )

        # Modality Gating Network (Learns adaptive modality reliability weights)
        self.gating = nn.Sequential(
            nn.Linear(embed_dim, 32), nn.Tanh(), nn.Linear(32, 1)
        )

    def forward(self, modality_tokens, mask):
        # modality_tokens: [B, 4, embed_dim]
        # mask: [B, 4] (1.0 = present, 0.0 = missing)
        B, M, D = modality_tokens.shape

        if mask.shape != (B, M) or M != self.num_modalities:
            raise ValueError("Modality mask and token dimensions must match branches.")
        if not torch.all((mask == 0) | (mask == 1)) or not mask.bool().any(dim=1).all():
            raise ValueError(
                "Each case requires a binary mask and at least one available modality."
            )
        modality_tokens = modality_tokens.masked_fill(~mask.bool().unsqueeze(-1), 0)
        if not torch.isfinite(modality_tokens).all():
            raise ValueError("Available modality tokens must be finite.")

        # Add Modality Type Embeddings
        tokens = modality_tokens + self.modality_embeddings

        # Prepare Key Padding Mask for nn.MultiheadAttention (True = ignored/masked)
        key_padding_mask = mask == 0.0  # [B, 4]

        # Cross-Modal Multi-Head Self-Attention
        attn_out, attn_weights = self.mha(
            tokens, tokens, tokens, key_padding_mask=key_padding_mask
        )
        tokens = self.norm1(tokens + attn_out)
        tokens = self.ffn(tokens) + tokens  # [B, 4, D]

        # Learned Modality Gating (Which modality drives final prognosis)
        gate_logits = self.gating(tokens).squeeze(-1)  # [B, 4]
        # Mask out absent modalities with -inf
        gate_logits = gate_logits.masked_fill(mask == 0.0, float("-inf"))

        modality_importance = F.softmax(gate_logits, dim=1)  # [B, 4]
        # Zero out NaNs if any
        modality_importance = torch.where(
            torch.isnan(modality_importance),
            torch.zeros_like(modality_importance),
            modality_importance,
        )

        # Weighted Fusion
        fused = (tokens * modality_importance.unsqueeze(-1)).sum(dim=1)  # [B, D]

        return fused, modality_importance, attn_weights


class EvidentialSurvivalHead(nn.Module):
    """
    Dempster-Shafer Subjective Logic Head with Dirichlet Distribution.
    Legacy concentration diagnostics; uncertainty interpretation is unvalidated under censoring.
    """

    def __init__(self, in_features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 32), nn.GELU(), nn.Linear(32, 2)
        )

    def forward(self, fused):
        # Evidence non-negative parameterization
        evidence = F.softplus(self.net(fused))
        alpha = evidence + 1.0  # Dirichlet parameters
        S = torch.sum(alpha, dim=1, keepdim=True)  # Dirichlet strength

        # Risk Score (Relative expectation of survival event)
        risk = alpha[:, 1] / S.squeeze(-1)

        # Epistemic Uncertainty: U = K / S (where K=2)
        epistemic_unc = 2.0 / S.squeeze(-1)

        # Aleatoric Uncertainty: Variance of the Dirichlet distribution
        aleatoric_unc = (alpha[:, 0] * alpha[:, 1]) / (
            (S.squeeze(-1) ** 2) * (S.squeeze(-1) + 1.0)
        )

        return risk, epistemic_unc, aleatoric_unc, alpha


class CancerFusionResearchPlatform(nn.Module):
    """
    Research prototype architecture:
    - Gated-MIL Histopathology (WSI)
    - 3D Volumetric Positional Transformer (Radiology MRI)
    - Pathway SNN with Sparse Concept Bottleneck (Genomics RNA)
    - LayerNorm Dense Encoder (Clinical)
    - Multi-Head Cross-Modal Co-Attention Engine
    - Dirichlet Evidential Head (Dual Uncertainty)
    """

    def __init__(self, rna_dim=1000, cli_dim=15, embed_dim=64, max_slices=32):
        super().__init__()
        self.wsi_encoder = GatedAttentionMIL(in_features=768, out_dim=embed_dim)
        self.rad_encoder = VolumetricSliceTransformer(
            in_features=512, out_dim=embed_dim, max_slices=max_slices
        )
        self.rna_encoder = PathwayGenomicSNN(in_features=rna_dim, out_dim=embed_dim)
        self.cli_encoder = ClinicalDenseEncoder(in_features=cli_dim, out_dim=embed_dim)

        self.fusion = MultiModalCoAttentionTransformer(
            embed_dim=embed_dim, num_modalities=4, nheads=4
        )
        self.head = EvidentialSurvivalHead(in_features=embed_dim)

    def forward(self, wsi, rad, rna, cli, mask, wsi_valid=None, rad_valid=None):
        if mask.shape != (wsi.shape[0], 4):
            raise ValueError("Availability mask must have shape [batch, 4].")
        if not torch.all((mask == 0) | (mask == 1)) or not mask.bool().any(dim=1).all():
            raise ValueError(
                "Each case requires a binary mask and at least one available modality."
            )
        inputs = [wsi, rad, rna, cli]
        image_masks = [wsi_valid, rad_valid]
        for index, tensor in enumerate(inputs):
            keep = mask[:, index].bool().reshape((-1,) + (1,) * (tensor.ndim - 1))
            inputs[index] = tensor.masked_fill(~keep, 0)
            if index < 2:
                valid = (
                    _item_mask(inputs[index], image_masks[index])
                    & mask[:, index : index + 1].bool()
                )
                if (mask[:, index].bool() & ~valid.any(dim=1)).any():
                    raise ValueError(
                        "An available image modality must contain at least one valid item."
                    )
                image_masks[index] = valid
                inputs[index] = inputs[index].masked_fill(~valid.unsqueeze(-1), 0)
            if not torch.isfinite(inputs[index]).all():
                raise ValueError("Available feature values must be finite.")
        wsi, rad, rna, cli = inputs
        wsi_valid, rad_valid = image_masks
        # 1. Unimodal Encoding
        f_wsi, wsi_concepts, wsi_patch_attns = self.wsi_encoder(wsi, wsi_valid)
        f_rad, rad_slice_attns = self.rad_encoder(rad, rad_valid)
        f_rna, rna_pathways = self.rna_encoder(rna)
        f_cli = self.cli_encoder(cli)

        # 2. Modality Stack [B, 4, D]
        tokens = torch.stack([f_wsi, f_rad, f_rna, f_cli], dim=1)

        # 3. Safe Cross-Modal Fusion
        fused, modality_weights, cross_attns = self.fusion(tokens, mask)

        # 4. Evidential Prognosis
        risk, epistemic_unc, aleatoric_unc, alpha = self.head(fused)

        explainability = {
            "modality_weights": modality_weights,  # [B, 4]
            "cross_attns": cross_attns,  # [B, 4, 4]
            "wsi_patch_attns": wsi_patch_attns,  # [B, N_patches]
            "wsi_concepts": wsi_concepts,  # [B, 16]
            "rad_slice_attns": rad_slice_attns,  # [B, S_slices]
            "rna_pathways": rna_pathways,  # [B, 16]
            "unimodal_tokens": tokens,  # [B, 4, 64]
            "aleatoric_unc": aleatoric_unc,  # [B]
        }

        return risk, epistemic_unc, modality_weights, explainability, alpha


# =====================================================================
# 📐 COMPOSITE MULTI-TASK LOSS SUITE
# =====================================================================


def continuous_cox_loss(risk, times, events):
    """Batchwise negative Cox partial likelihood with Breslow handling of ties.

    The public interface accepts positive risk scores. Its logarithm is used as
    the Cox log-relative hazard. Censoring is indicated by event=0.
    """
    # Descending times put every risk set in a prefix of this batch.
    order = torch.argsort(times, descending=True)
    ordered_times = times[order]
    ordered_events = events[order].float()
    log_risk = torch.log(risk[order].clamp_min(1e-8))
    log_risk_sum = torch.logcumsumexp(log_risk, dim=0)
    # Breslow ties share the denominator containing all rows with that time.
    _, tie_counts = torch.unique_consecutive(ordered_times, return_counts=True)
    group_end = tie_counts.cumsum(0) - 1
    full_risk_set_logsum = log_risk_sum[group_end.repeat_interleave(tie_counts)]
    return -(
        (log_risk - full_risk_set_logsum) * ordered_events
    ).sum() / ordered_events.sum().clamp_min(1)


def modality_orthogonality_loss(unimodal_tokens, modality_mask=None):
    """
    Mean squared cosine similarity over available off-diagonal modality pairs.
    A single available modality contributes zero; missing branches are excluded.
    """
    B, M, D = unimodal_tokens.shape
    available = (
        torch.ones((B, M), dtype=torch.bool, device=unimodal_tokens.device)
        if modality_mask is None
        else modality_mask.bool()
    )
    if available.shape != (B, M):
        raise ValueError("Orthogonality mask must match token branches.")
    safe_tokens = unimodal_tokens.masked_fill(~available.unsqueeze(-1), 0)
    norm_tokens = F.normalize(safe_tokens, p=2, dim=-1)
    gram = torch.bmm(norm_tokens, norm_tokens.transpose(1, 2))
    pairs = available.unsqueeze(2) & available.unsqueeze(1)
    pairs = pairs & ~torch.eye(M, device=gram.device, dtype=torch.bool).unsqueeze(0)
    return gram.square().masked_fill(~pairs, 0).sum() / pairs.sum().clamp_min(1)


def evidence_strength_penalty(alpha):
    """Mean evidence fraction (S - K) / S for K Dirichlet concentrations.

    This is the pilot's existing evidence penalty. A censored survival uncertainty
    likelihood and clinical validation remain separate research tasks.
    """
    strength = alpha.sum(dim=1)
    return ((alpha - 1.0).sum(dim=1) / (strength + 1e-8)).mean()


def evidential_kl_regularization(alpha, events):
    """Compatibility entry point for the legacy evidence-strength penalty.

    The events argument is retained for existing callers. This function computes
    the same event-independent quantity as the pilot implementation.
    """
    return evidence_strength_penalty(alpha)


def composite_research_loss(
    risk,
    times,
    events,
    alpha,
    unimodal_tokens,
    rna_pathways,
    lambda_edl=0.01,
    lambda_orth=0.05,
    lambda_sparse=0.005,
    modality_mask=None,
):
    """
    Pilot composite objective with an evidence-strength penalty.
    The lambda_edl and l_edl names are retained for existing training callers.
    """
    l_cox = continuous_cox_loss(risk, times, events)
    l_edl = evidence_strength_penalty(alpha)
    l_orth = modality_orthogonality_loss(unimodal_tokens, modality_mask)
    per_case_sparse = torch.sum(torch.abs(rna_pathways), dim=-1)
    if modality_mask is None:
        l_sparse = per_case_sparse.mean()
    else:
        has_rna = modality_mask[:, 2].bool()
        l_sparse = per_case_sparse.masked_fill(
            ~has_rna, 0
        ).sum() / has_rna.sum().clamp_min(1)

    total_loss = (
        l_cox
        + (lambda_edl * l_edl)
        + (lambda_orth * l_orth)
        + (lambda_sparse * l_sparse)
    )
    return total_loss, {
        "l_cox": l_cox.item(),
        "l_edl": l_edl.item(),
        "l_orth": l_orth.item(),
        "l_sparse": l_sparse.item(),
    }


In [ ]:
print("[*] Processing Phase 2 Pan-Cancer CSVs...")
root = Path(CFG['dir_multimodal'])

def required_csv(filename):
    matches = list(root.glob(filename)) + list(root.glob(f'*/{filename}'))
    if len(matches) != 1:
        raise ValueError(f"Expected a single {filename} within {root}. Found {len(matches)}.")
    return matches[0]

cli_path = required_csv(CFG['clinical_csv'])
demo_path = required_csv(CFG['demographics_csv'])
rna_path = required_csv(CFG['omics_csv'])

print(f"Auto-Found Clinical Data: {cli_path}")
print(f"Auto-Found Genomic Data: {rna_path}")
print(f"Auto-Found Demographic Data: {demo_path}")

# Bulletproof Pan-Cancer ID Standardizer
def standardize_id(df):
    for col in ['Patient_ID', 'bcr_patient_barcode', 'patient_id', 'Patient ID', 'PATIENT_ID']:
        if col in df.columns:
            df = df.rename(columns={col: 'Patient_ID'})
            return df.set_index('Patient_ID')
    raise ValueError("Expected a named patient identifier column.")

def select_reviewed_clinical_features(frame, approved_columns):
    """Require an explicit prediction-time feature contract; reject outcome proxies."""
    if not approved_columns or len(set(approved_columns)) != len(approved_columns):
        raise ValueError("Set clinical_feature_columns to a nonempty, unique list reviewed at the prediction time.")
    missing = set(approved_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"Reviewed clinical columns missing: {sorted(missing)}")
    blocked = [name for name in approved_columns if any(token in name.casefold() for token in
        ('last_contact', 'follow_up', 'followup', 'death', 'vital_status', 'survival', 'os.time', 'pfi', 'dfi', 'dss'))
        or name.casefold() in ('os', 'event', 'time')]
    if blocked:
        raise ValueError(f"Outcome or follow-up fields cannot be predictors: {blocked}")
    selected = frame.loc[:, approved_columns]
    if any(not pd.api.types.is_numeric_dtype(dtype) for dtype in selected.dtypes):
        raise ValueError("Reviewed clinical predictors must be numeric before model preprocessing.")
    return selected.copy()


cli_df = standardize_id(pd.read_csv(cli_path, encoding='latin1'))
cli_df = select_reviewed_clinical_features(cli_df, CFG['clinical_feature_columns']).fillna(0)

rna_df = standardize_id(pd.read_csv(rna_path, encoding='latin1'))
rna_df = rna_df.select_dtypes(include=[np.number]).fillna(0)

# Deduplicate clinical and genomic indices to ensure robust 1D vector indexing
cli_df = cli_df[~cli_df.index.duplicated(keep='first')]
rna_df = rna_df[~rna_df.index.duplicated(keep='first')]

# TCGA-CDR BRCA outcome extract from the official GDC Table S1.
# DOI: 10.1016/j.cell.2018.02.052. Embedded to keep this notebook self-contained.
import base64
import gzip
from io import BytesIO
from sklearn.model_selection import train_test_split

encoded_outcomes = (
    "H4sIAAAAAAACCpWdSZPkOnKE7zKbX6FrP4kAN/CYtVd1de1d20WmZQ5z0GKyuejfKxmRCQ9HRFfzmfTckz0fs5hcQCyBwL/9"
    "+//+y//869//9tf/+vu//Nu//u+///d//PXb3//vf/767f55////9Pe//edfvz1cXK//6cbZ8/P632Fj/7+cHf6Xv/zDy+nl"
    "7o/+9I/dbvfz28nT6e5b923ohvlb+pZKV3jL4rfXwLsxNMJvjnga5iE0wr8DH0pj/3j3zzuRwx7D1Xr4dY9eyEYP6Hi7R186"
    "cyxzY+2X6x7J7JEac3u8/LFbHnf1cLqetMV36Y9d93xyxPO4eCX0rJ74fvZK6Hk97D6n0Ai/wK+c+tAIv8SBDF4JvcI393No"
    "hNd7a+pHr4TW+2pIk1dCD3dI+rZMM2l8SX7USzJkr4Te1Z+Xlik0wu+PeBmzV0IfjuhYBq+EPlZ0HLwe0bxHhztcuXEIjfCH"
    "44kb94/Tvijo8uGzO3GC1wPJfRlDI/ypPilzt4RG+Ev99jxMoRFeS7U8yY/zRvjr8aemJXerDXPHW4S/1WNfT0pkhL/X+2Qq"
    "U2iEf1S8W5bQLH76HY/CvuDYH20/DbxF+C1KqDQ2Fl7Y0x/HkzPvT1761pdsPxN6f0T7YZFTYSz+8nqn55L60AjHTTYtY2iE"
    "4ybLpW8sPp7n4y/I+hKxFu9h7syyhEb4z3qrjfJqqxp/+6u5XCk0wt+++MFqhOPOnHMJjXDcmdPch0b4J/A0NRb+2rP6is6d"
    "nnZnhOP93I0pNMIz8DyHRnhf8aK/zxnhwxGfp9mrRc9/4KIuQ2iEo9zuSgmN8Hu8wQ6vLVh42s/rQ9hPWmFxRvijeUVOoRH+"
    "hAdEahTeCH+ueFq60Ah/wUWdptAI/wl86EMjvD5+y1S8EvpWH+xSBil5x/64FZ/2+vTNUglolFA8eUseQ7P484spNebQCMdJ"
    "maYcGuH1bVn2f91q+DOfcWL6aSUnPS/rRrwDSqWxlzOZ+4W3CMe56Yc5NIujGTD2vVdCa/kySs25UUJzfW+MI2n4E19Qtoxa"
    "JjojvJYtUx68Ejri5yWvhE4VnUevhM6omievFv04rY/M0klFYTp8Ds/ExxmKzcUroeebq7ofF+YWGkMjvLZpJik1GiW0tmkm"
    "aaY1Sug1DqIbQyO8tmnG/R/e17TlO+tnQr+jfVBI43N8u7UVrvgPFHqjV0Jfat0wrw9QowZNF7UondPgldA3PHfFK6EoK4o+"
    "7s4Ir1WSSSpEjVr0spYTWmtplNC0qbkmaH2SR6mxNkooHs8ueTXo2mFQ7zRtexqLbok8oqMiz1NohNeOipSXJIWwvI1lK/4D"
    "p7iGvVdCz/BcZ9L4m89x6N36QsiL1LTNFuEXwLNXg/bfT9Ggl5acN8LPcFK6qbHo0Pv32rpIXb+2oiZpS2GDYDQs5rkLjfDa"
    "sFiSNNTnwX4mFLWaZfJK6BtaCCmHRvh7/YGDvrOdEf6B87GslYLSZdog+NOU6Tk0i3+gu28cpsaiqzM8vaGimr0Siqqb1K0b"
    "JbRWTyZpxDRq0ed6yLPcSI0SWouckjuvhGYcQO+V0H7Te3BeG3e4+ZO8GbwRjj7JTl86zgi//KKa7i6e7HFl/sASGuH1ZbtI"
    "JQYafvsZCtikd7Izwnc4GOn98kY4SteumxuLjweVLLk7GjVo6s82vc0FPT8+jlMapNYx2c+EXqDaNnol9BIv/rUfcZRer/Wz"
    "KKH1IpZl7bya5b1YpN+rLPYU56neUItUABslFMfaF6+E4ljlsW+U0Cug+t5J9jOh6NiekldCb1DTn71aFK+fSW7xRg3aX39g"
    "lGT0Sujnptflit506AbtvRKacAC9V4s+XZjejvVlIAVE/WzQIde2ziCDVUNa7GdCJ3RxDF4t+lwf3kH7xVkJPUHfa/FK6CmO"
    "dfJKKIZ6huSV0HOU7YNXQut5HeW6Q6MSZpzMQRSvFv1EH6V0jDRKqOmfnLwSWl+xfT+RRsc7vb5uvRmnV7zndWiiavzNOBDt"
    "PWEl9APd05PXI1r2JfhUO/Xo/wipHXm9VM0aJfRhO/qIn9MIYU+4XWavhL6Yft4hNMJrRbVIu6tRQmt1cpGiolFC66mfpYEy"
    "a3F8/EzoZ3gG3MVf2RndKfJUN0rohNMw5NAIX1C9H7wSivpBL4Pz3ghHM6brh9AIRxkiX9cooShDZOi8UUIvt6OIKZCCoFFC"
    "b1DUjF4JrZ0ei/RoNUooHrRu8EooHjQ5mY0Sin5vGTCAxnca+r3l/obG+E9UwDqvhL796ufF34x2Vq/hDs4sXtJvC7KSUZYs"
    "Xgntf/9tA97PhTT8OWVE1V06TL0RPm0uSYsJ9miEMDy9aZxCI7w+vaUkr4RemHbDEBrh6OWU5kWjhF79/jJcb34Uy01t5qc8"
    "NxZftNs6kPy7J7LUh3d9L+wLexmzWH713JQHi3/5SilP267v85+7Ci8YI+8WHSSYj1vxIb9T8b9HdWirbhCMpzdLFcWbxZfu"
    "txd6SfT70qFkwgbBGac3eyW0R6kweSV0+P1R4jHPGtbljPDp9984b77Bl+VPlbDL7s/hJ9sP5HR71WZBD86Yc2iEowdHgzqs"
    "xX/g8vfn+PpPvWuWOu6xSPdjo4T+MHftFBrhd5tqxcsj3rqJND4Dv6wCiBL6gvrS4pXQV7wAZ6+EvoVFWHywtbKNCpiA9YPb"
    "49Nf4BDcpc138C5vPg+7YfOrejehndx5JXTeVOrvFnTAZq+EnmxuQ+xMr6pcT29H/PSPXe4xyCJt2EYJRS1/KV4JvTSxdiU0"
    "wq/q212GHKDuXljxk4Juq1miAYbDZ1FCv5sO+Rwa4SasRvurnFn8woxpacPIGeG1czRP/SQVjYk2CL7Ag6/RvM4IR3d3GfvQ"
    "CP9uohu70AhHAN8ioXuls58JRezQGrsQmcUfTae7jmk7I/z6V8PWooTWKuPaRbZG4832s0H7k3odlzJ7teg5btVFnidWi14h"
    "6EoKlEYteo+uMok7a9Sijw+4+UevhOJVI/cjNHqs+pcfeDNlr4Tip0kLpFGLvqH7Q+oo0PAg3lDMyohpo4TWYlZj6hu1KOIo"
    "ZmmrNEroDUKqBq8GHT7PMdbQeTXoeH6FAjx5JRR9GVLnbdSi789bn4bxHZM05Dls1KBTfkXzopBGV2zK7+jBnr0S+oHIoo40"
    "/ObrV4SzFK+EvuEgeq+EojNWalGNWvTuHletEYPNr2YORueVUHSs2f8MUu7xgEvnZ6OEoiN2aISwJ4yHNELYMx6SRo7Y3b6C"
    "0aMt2TVC2McmbKj1utn+R8i07ZswwLoYcXeWsDsMfhqJWdSQBiMxiziUbCRkdze/mugjSqipxQxeCb0FOnsltJb3o/2PkKdN"
    "p373vA172YRdoAarIUishF6Y6HOvhN78alZOeHVMBUmnEbASilMu7b1GCb3DtxavhJog5s4roQjzkVKiUUJft10CxOykRgjb"
    "VhRcbCsKEFn3JYaousNUAIjF3hGxqJPQWAm92/SH39HHL7WtRglFF6H9j5Btj9b7tkfrfduj9f5zG4a75BC+TUqoie7qvB7R"
    "+/WNcPvFrAI1wtH9UrS30hnheLS6XuILJxmeNluE13lG+7aoDpO0RnidyZaXlHReVz5uuQJE9kDHTScDid4IN8+wluvOCMcc"
    "oFki3Y4jpeuWGuG4oL1ObHFm8Rt0wUrDplFCEe8lX9UooQN6C6Vp5Y3wsWlzVg3P9w1qCUkqN94IRzi6lL6NEoruhtJ5JXQx"
    "49hyWQ5zSLBF+A4nb9SIR/uZ0BMTi5hDI/zU4HNohJ9tnjEj+DneeP3YWHx18KJe5MR5IxxBVNK9BI2/3YTJ67QzZ4RjSnnR"
    "W88Z4eZNUvrQCP9h3vFDaAZfH+/jW2WUfmVvhI8mDmsKjXCMRfbdLJ1NM20QXBBao9PfnRG+mJmCU2iEY8gipdxYdFlTRrtK"
    "47O9EX6Ku1JqgN4Ixz0/yvxVb4Sfm0pWDo3wCzOrIIdG+KWprSyhEX61ueNxxb/fIyQweyUUs2kX6Tn0RvijeZRyaIQ/mXPe"
    "h0b4s3k2vBKKwQ+Jcmv0iD6ujbSrTT08gtaSokh7slFCMSt6kZ52bxb/GPEC/6LlskfTVUY/1+iV0B4dmYtXg46Yyj+XRgi7"
    "N/2zKTSDzz/nOuP/q3ksT+uEsAfE7OhvcUY4powuWoNyRvhTPb8S0LKqFDJ1k2g0tkaJBfFGOCZH9mkKjXAE2khmgkYJfTVT"
    "wUtohJvBOYkEmKW4wAbBaK0tep85IxwzKyYJpR6HgTYI/qx9hnmSCRhr3KSccPMPdo+fnakRlNAIT+jDkm4Vb4TbyZRa/z9G"
    "Shy3CO9NxHAXGuGDiUFNoRk87WqB05dOo1Y0ErVIB/RqhCMsJi8yda8U2iDY9CLNOTTCkewmyTiSN8LRZFvmFBrhP/7MM5p2"
    "aLItMu3EG+EYWdJ6ujfCTV4IqRt5I/zRDP2l0Aiv5cuo8fpTsZ8JfTYjsl1ohNfonpT1wVxNyy78A+2BVCVFOse9Ef5qGtZD"
    "aIQj1nmSaVLeCH8HnpfQCDd9Q3kKzeAZ7+6hSKYDb4R/N82gEhrhNWYsFxmaTYPMXDFbhGO0eJK3ijfC70yH7xga4fcmpYLU"
    "1kctXbBFeL3d5WVrhTATu60lhDPCn8ysiy40wp9NeTGFRviLmYeZQyP8J45dGnbeCH81396HRjh6skZ9MJ0R/m5evUNohH8a"
    "PIVm8RFpTHSU0hvhydxaXWiEZxQYfQmN8AGPdDeGRviIlpLcsd4In/DQjVrDGA51NfMPdo9bzOLvpG4x6BWVLTXCrzAnbX2M"
    "h+MAeN0k+ubPFHf59jtqSYNmSzoObZt/oD3wTu2kAeyNcHSDLpp5wBnhJqPYlEIj/N4kICuhEf5o2p19aISjj1sntnoz+PiI"
    "6kOWTlvNkiIbroNAdjAd96lvLN7jAb9XJ1Iai/d4RM02SzRxOnxu8ZNpXzW8Tsj+NCWNUJbKDLYIR9uulwtkLf4DI3LASUXG"
    "G+H16VpkIkuRlF71M6GldrjKjTtLqMz6OT6QpZ4YeQihMV67f0ovtUFvhJ/Ub1+G9WDznObjVvwHTpEW6+seUsXPEcijc8U1"
    "NUzdIPgSsUQyo9VafDRXCKJImkoyHXovzT/QHgg8STpf2xnhteAZ5/X87Ks+9jOhNXqr18RT3gi/Q9tp7rXek3mL8HsT4JND"
    "I/wBF1Yn7Tkj/BG9BGUMzeJP9dUwzxIP6I1wxCl2JYdGON4kev2zTlU3W4Qj6mjtiF2f/q7jLcJR+xwkeY03wn+gpNFwYGPh"
    "PflUi8vSydwsb4TfI35N6u7eCMeF1d+X5zLzFuGPJlZ41mDhxgh/xnSOab0NcyfFgtki3GS+kPabtfjk1MplWas16zdLndVs"
    "EY7YJJ2E483ib0gKqPPmrYXH82aiuftFnl2t3WCL8FfcPBJ65Y3wmlFr7eFKhxnP9TOhNah7kr88ygmvnwn9QKEnr7vcyWCp"
    "2SIcqXSyDO6NKc3HLTWLv3fmlSAtpuU49dj8A+2R8EaTQbVBhrOwQfBQJ5VOEt0qtcP6mdDaeZvXOR5rHW/oeYvwGX29RWsB"
    "Oq9n3YoufjJpZHRmszfCkaBGE/p5I/za1LOm0AhHsKdW87wZPF//xJMxy0tSA5VkS83g/SfyvaQxS91kOG5E52bo0DTSgRpW"
    "QpNJjjyFRng2na9ynTSXmtkivODloSNMzghf8MJebxZ9Z9TPhJ6YXrocGuGnpsE1hHbEr9YwLASBLBo04Ixw010kVVlvhL9t"
    "GWtS9MO8rL0S+mmOeQnN4ie4RzRDtzfCk4kdKKER3pusTX1ohA+mjZVCI9w0n/V5NOaeCNljwq/9OpZF8dkMdiyhEV7QLzIt"
    "oRFuEolpb6UzwncmLU4OjXBkrZFp7o0SerYl15+iF8gvqVN6nRFu5lroJDNnhCMhi/QLNkqojd4soRFuQqTmKTTC702Uxxwa"
    "4Yh5ylpeOiP8kSZ+rC+7buEtwp9MDEkXGuHPpvM/h0Y46n86R8wb4a8mmeYSGuFvJtIghUb459YpMYKfdmYeahca4QkJ09Kh"
    "j6HnLcJ707O4aBDIwFuEo7DQRMTeLH52ZoqiKTTCz7eGbChummtDCo3wq63T5hW/3jI1VNHvpkrklVATHVxSaITfmxQgQ2iE"
    "YxBdHvlGCX1EbEAjhD3/Li5AMTxgQ9eHRjjGtTutlDgj/N3U8abQCP9ES3X0atHzzqT69Uoohpyl1tAooaYbb/RKKF7WS++V"
    "0BnHqnHLzghfzLO3hEb4zvYMdq0SeoJchaNXQs+3JEdQ9NpkpJ5Cs/hlrS0u8pw1SqjJQFa8WvSq3g+DdryyEtqb4y2hET6a"
    "IOouNMInnOPZK6Ez4kmyV0JRG8sa+emM8J2JX02hEX6CDqXJK6HIoCdzURolFLnu5Ew1SijGd+fildBbHEAjhN2h9Tp5JfQe"
    "8yUbIQxFdBq8EvpoEs2k0AjHeKiukcFKqBnc1IEgZ4S/Y2r64pXQDzPEPoZm8af3LfMFBH3rccUKadgKesPqCdKOgca4yfah"
    "EUfOCP9zZfHbDo9o8mrQRCl4voywU/zSJNHwSqhJNJ69EnptVozpQyP8BkOqmTQ616nUsYp9fbFvLN6j9oUXKXnKcZpF3SQa"
    "XeFZx8fzMe2p+Qfa4w69Tmjq67Bt2OhPpbaGlpxI4x/waKLndN7MceaC+Qfa48kckrZ9WyMcPcqdtrOcEf6CUzSQxodv1pjR"
    "G9wZ4Vj9SIchxfTX4h/sHucYZEo6OdEZ4Ziuo3MBrIW/4PzZ9EXnxuI9cIpGHbM1Fu+Bs6Qx79biPTC3WhaygMb4m7mLhsbi"
    "PdCdOmrCU2PxHrXTfuxloreGGfa/wC9qL/xcRtIYr4GYi5TH0BhHz7qmB7cW7zFFIyv67C+/Oq8XGOqeetIYx2D0NC2NxXuc"
    "4u7TwCpj8R5nSGQuLXFr8R5IJ5KlF9lavEcdxu5njUCdj32K+OR2ujIZ54fG4j1uzFpomhjjmKUNn9xOt7jVdZzZWLzHD/Ta"
    "6Tu7Py5fgU9uJ1PuyJiTtXgPE0k+5MbiPVAsTKU0Fu2RMVBRDlEcR43xjOlRI2mM93izaGYdY/EeA4pCaRqL6UM19r/aqWCY"
    "S55ya/EeCCHRcsFauEcx8UrajHJGuKlQa0+esfAP3GKynxQ8jRp0yEjDJ3+8UUJ/mkzEQ2iEv25ZxnJFRyTbHqVDrlGLItn2"
    "LINljRp0eipbOymmJzPlsfNq0HLxgaJaS2pSQvGzpHeyUYuaVVZkgLxRg55ggluWmX6NHtDTss6w+v5V5jlFMONn6r0Sikhh"
    "mVDSKKF321FECEswTKOEPmz/VgS8yasd2j4wir/8/kT93H6cr79C4z/+tv2b37ejH2axoyk0wtHbJFFDjVq078ySpV4JRe+j"
    "NHkaJTTjAHqvhPa/vVr9sP3bMBlVioZGCUVOvPmwqKj9TGgx02W8WvTqfEtKBkURIyEji40SerllrQNFr7esSqDozVfpVhT5"
    "biYUeCX0dktKeUV/bD9JKHx0HUlWQu+3fmueXresg6WoeaCTV0JNHqfJK6EfW39WRmb3QZ7KRi06I/usRB03SijW7JJQu0YJ"
    "Hb/KEKWIeTvPXgndbelVVhQrUcgkwkYN2v+Yv2gSJJseXvGy9fSXq02rECj6uLWYKFdPdkEQpwZdTMVDemkaPaBnUk59bF36"
    "VvHPLZU6QQcMSshoWqOEJnSWT14JxSBVHr0SivmQffJKKFZQkUwyjRI6bVlGStF5y3p4giI9x2+PFek5dMGvRgnd4YmVKSa6"
    "/OXxM6EnJj9ECY1wLOOik0tZCf3lsnaihJ5vWYlT0YstbxlFTVoCXa3NGeFXW5ZhUPR6y+IOiiJ+Rmo/jRL6fUvrTdHb7d+K"
    "RGkSP9oooXdbxq4URZ4/KWMbJRR5ECTApFFCX8w8aa+E/rRrHzm16DuSeEofW6OETlsSMSiKh1z6rBsltCAqofdK6PKrkca2"
    "CaG4nZfildCTzY/jO/r/JEUxND4IkzcweyUUi1tJSxMaf/PllkWYFDVxaoNXQr9vWV9KUTxnMi7cKKF4zmQeYqOEmjAZzTTi"
    "jHDMIlkWr4SiriBf1Sihz1sS/Cr6YqYseiXUjOqOXgl93RK8oOibWQFiCY1wZGo9TK08anyXodYsr5FGLYpFNbVHHBp+80fC"
    "vI9MGuMmEKf3Sqip43ReDZqxvlGbt1aUUDye8vZvlFA8mnINGiUUGSSW2Suh11uq8YoiekLvGW+E3259iPL8Y2ulMM8m4GLx"
    "SigiKZZGCDOLrsxeCcUEdBlObJTQ1y0tU0XftmQYXtH+czQzWMfQCMfLVYr9Rg06fKYtq4yv6Nz/NBO9vBL6xp2DMgh+/Egg"
    "lreVvuhGD+j5ukp1N25dm0xxBNBprcIb4bNZB3YOjfCC6sDslVD0fOskK2+EI7YqZ5lfO9rPhJr1ngbNNjkeN9piT3c4Nd/d"
    "hUY4EpdLFa5RQmt5ViSqqlGLIr9FGjSe0hnh9yYn5RIa4Q8mkm0KjXAz8qMhC84Ix0x1HRvTBDdJlw5NWDVUabte0xIa4SYY"
    "PaXQCMf8nD6V0Ag3KX01UsQZ4W9bZlAo+g40eyX0w2QrXkIjvKaL0tmEScZH62eLjt3W/AeKZ5NHVb50sJ8J7bcsCqAo+u8k"
    "PrZRQsct68QoipBPaSI3Sig6OSQbSaOELluWcVB0Z6aceyX0FIu/DF4JNY2ZySuhWFJGKlONEnqB91nxSujllpk/il5tGcFQ"
    "FCFyJqKrC8K5FL8xgfxdaIR/N8u9eCX0dkuXn6KY+d+sgCtK6D3Fca4hTZoZGluEP+CJGL0SarKA6KQuZ4Q/mxn/Xgl9MVGt"
    "X0apCY5JhLnXZFTOLH5iEo33KTTCk1lSpw+N8MHE+3y55p/iI2ZvDF4JnfB2n70SirUNukQaVjNOzNpSmvDFmcWvzewezQFY"
    "UsdbhF+Yby+hEX5pErJ2oRF+tTXRmOImIDaPoRF+s7lGdY1WV+m60Ai/NRlrvBJ6t2VpIEXvzXJRY2iEm/iV2atFb+0axiU0"
    "wlE9L7IMVpo0QgdbhBezEkYXGuHo8Jc2XaOEmpGy4pVQ5CXupQfHG+E1Tq/MGjWY7GdCL00i6z40wq9MwuY5NMKvTf601Fj4"
    "wCO7VdYMeDKVs34m9JY6mCTvVc9bhD+bsKEhNIPnhxoXWqTJOUqihfqZULNcjGa3d2bw/swsjaqPtrHopIwYIs7S9duoRef6"
    "ClnkndEoodlMKi+hEY7JSJr1zBvhZg76vIRGuDmHh6VaWzP4gtwpX3Xcni/7635njlvbec4IH0xLqwuN8HFreKDiNXZ4mjXz"
    "SW8/E1rMiuhLaISjbyDpNA5nhJtZ8Dqy5ozw060JCBTHazvrcgTOCMcyrZo/xBvhmDy/Jr5I60JK5jOhl4gdl0YOK6FYME3T"
    "hrASem1SK3q16OMn5giNXi361Jl8QKSuUBA8m+exbyzeo//dRFvFBpOt1SuhqCPmRggzawIsXgmdTQ/xHBrhOzNJagmN8BpW"
    "r+lBGyXUdnANoRF+9ruF0RQzCxJrrcMZ4ViSU26uRgnFQunyKm6U0KvN3YiCo2I6HCZ9tWbwnCc8nbKO33x4PI8bBM/o8Bm9"
    "ElrM9LkxNMKXrRlWFDflYVlCI9z0l+qN6YxwUx7qn3dG+LkpPpfQCL80OTKX0Ai3Sznk0CyOyBydc9fLyH79TCjStqZGCEO+"
    "makRwmZzSabQCC+oeTdCGOasa54yVkI3hbIJipXcv4pNFPSmThNbZDF6zeBdPxOKblxNxs9q0P7KTFMYS2gWf1y2TgFVHKsX"
    "SjOwUYsibkIn1Tdq0BG9vV/N0FYUiZukrtyoRbGOXpGqTKOEmvXBGrHYT0y7lqKmUUKRE1sqRo0adEIkcZE7qtEj+rY2ht+2"
    "ROMo+r65Ui84etEP606QEvq5qY9zRW/SltAwRfOWwENF+1+t6ytK6LglkYai6LuS42vUovevuA8Wr4S+beq8FhRjHjI40yih"
    "SEghV7NRQj9p7C8yiz+YDEC6XkW3DLxFONKXrX22K6CZ3LBFeL81yYPiA37m7JXQESMqvVdCzUh2JnWVZcFRM0ky31TOW/1M"
    "aEFeVMlopOnyj58J3W1+Mh5OEMZUvBJ6arJyeSUU635LSdcooecmm/0vZ5sperElBETRy82dJYIjo4Bc/0YNmi/Q6z/NfWiE"
    "b+8PEtw8/MUroXj4ZeSyUYP257aRP4ZmcSxh81U48YqumaEq2nsldNgyTW9Fp+dlayyg4rjdpR++UUKR8UfavY0SipR80gXW"
    "KKFnJtqlC+2AX17vr9yp6ewdvRKKtej6IYd2xH/s8bM68TQVqTqP2ttcNwiuiZKmUZKD952sCmK2CD8xmbGm0AhH8s5RF1F0"
    "RviZyVGaQyP8wiTSGkMj/MpkCOxDI/za5FftQiMcY5yHtVmdEY4pThoO5o1wTHPKmkrcGeFmNZYlhUb4/dZUsive372ZCb8p"
    "NMI/TAM2h2ZxBPUmzZbojfBbE5xRQiPcrKSbltAMPpybfLy6zJMzg48PiFmVRkCjhL6jjpu8HtCru/3luUN2p6URi92fIDNB"
    "1+lSAT1vHfCb2/2Z+HgzS/B4JRSz0TRHL+sBvT3bl4LTT/QNTV4NOiMUaOiyV4u+2VVXSNt3geITvnnwatDlEfNFcu/1iO7P"
    "woCJS20OJlFCE9YYnrwSWlslRWr3jRp07D62BL8I+nG7JfBU0R82vDkdWmT1M6F3WyZVKIr8+jJ42SihD1vCf1Z0usC0Emlg"
    "NUroI5rHjRBmIuIXrwadMSXuq5aeop8me4mmCjl8dvfripspcV/MOBD02UzhG7xa9KXbMod0RcsFpsDIPdqoQZfHfku98PZx"
    "LTwHdDusDZRJ4olLsObKj+f9fZBQdAjY6AG9/76uW5NRyjRyxNY76wz9w4ek9M4In03CvS40wtH9JLV+aPvzFN+Zb+9DsziG"
    "Hfct3SU0wl+3Dpkq/maG8KbQCH/fEiirKAopeT80SigmjcrrrFGLPndbpp6t6DTVap5chsgIN+uYDF4JRdSbFuXeCP9u1jrU"
    "cYPWCLcxBik0wrHErQyDNUooFrGRQLNGCX3YEp8m6Os90voUr4Q+omXdeSX0aWuw94ovMxoYsy6kNB4+t8/gw/n6DGIZJZ0L"
    "Yy3eA606XSTaG+HnX7QC3B9Yi97bV5+0IgbffYqFGKxPYKn/xaBZti+Ttvhzv780nxfoUMpeCb38VQeYKKHodZH3XaMG3XV2"
    "SRWvFk3ovZARnEYJRS4SGUBqlNC8JXGPogiqluNrlFC8fKX13yih85YutZ/7i7szs2lk7ATaXtyfP/cPUv+MV+VE2uKv875u"
    "gfwEuZN+Hm8H/G3NUTCZpCOz1yP68sfuBFXn33XgKo46hrxGGz2g7+/7g1hMh9LsldDdlolun/szURCX0GdZEirprKl1oz11"
    "ugMiFPI4hfaXf/h/BdUqO83AAAA="
)
audit_dir = Path('/kaggle/working/data_audit')
audit_dir.mkdir(parents=True, exist_ok=True)
outcomes = pd.read_csv(BytesIO(gzip.decompress(base64.b64decode(encoded_outcomes))))
if outcomes['bcr_patient_barcode'].duplicated().any():
    raise ValueError("Duplicate case identifiers in the curated outcome table.")

available_ids = set(cli_df.index) | set(rna_df.index)
matched = outcomes[outcomes['bcr_patient_barcode'].isin(available_ids)].copy()
matched['OS.time'] = pd.to_numeric(matched['OS.time'], errors='coerce')
usable = matched['OS'].isin([0, 1]) & np.isfinite(matched['OS.time']) & matched['OS.time'].gt(0)
target_df = matched.loc[usable, ['bcr_patient_barcode', 'OS.time', 'OS']].rename(columns={
    'bcr_patient_barcode': 'Patient_ID', 'OS.time': 'time', 'OS': 'event',
}).reset_index(drop=True)
outcome_summary = {
    'source': 'NCI TCGA-CDR Table S1 BRCA extract',
    'endpoint': 'overall survival',
    'source_cases': len(outcomes),
    'matched_cases': len(matched),
    'usable_cases': len(target_df),
    'matched_observed_deaths': int(target_df['event'].sum()),
    'excluded_missing_or_invalid_endpoint': int((~usable).sum()),
}
(audit_dir / 'outcome_summary.json').write_text(json.dumps(outcome_summary, indent=2), encoding='utf-8')
matched.assign(usable_endpoint=usable).to_csv(audit_dir / 'outcome_rows.csv', index=False)
print(json.dumps(outcome_summary, indent=2))
counts = target_df['event'].value_counts()
if len(counts) != 2 or counts.min() < 2:
    raise ValueError("Too few matched observed events or censored cases for a stratified split.")
train_df, val_df = train_test_split(
    target_df, test_size=0.2, random_state=CFG['seed'], stratify=target_df['event'],
)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)

print(f"[*] Training Cohort: {len(train_df)} patients ({int(train_df['event'].sum())} events)")
print(f"[*] Validation Cohort: {len(val_df)} patients ({int(val_df['event'].sum())} events)")

audit_cohort = pd.concat([
    train_df.assign(split='training'), val_df.assign(split='validation')
], ignore_index=True)
if len(audit_cohort):
    audit_dataset = PanCancerDataset(audit_cohort, rna_df, cli_df)
    print(json.dumps(audit_modality_files(
        audit_cohort, audit_dataset.img_folder, rna_df.index, cli_df.index,
        CFG['max_patches'], audit_dir, image_manifest=audit_dataset.image_manifest
    ), indent=2))
feature_contract = {
    'rna_columns': list(rna_df.columns), 'cli_columns': list(cli_df.columns),
    'max_image_items': CFG['max_patches'], 'image_size': [224, 224],
    'selected_tables': {'clinical': str(cli_path), 'demographics': str(demo_path), 'omics': str(rna_path)},
    'configuration': CFG,
    'outcome_source': 'TCGA-CDR BRCA OS and OS.time',
}
(audit_dir / 'feature_contract.json').write_text(json.dumps(feature_contract, indent=2), encoding='utf-8')
print(f"Data audit saved to {audit_dir}. Review these files before interpreting performance.")

if train_df['event'].sum() == 0 or val_df['event'].sum() == 0:
    raise ValueError(
        "The pilot split has a partition with no observed events. "
        "Audit the raw outcomes and define a feasible evaluation before training."
    )

if CFG['num_workers'] != 0:
    raise ValueError("Use num_workers=0 while the dataset runs the frozen image encoders on the main device.")
if CFG['cache_frozen_features']:
    image_cache_mib = (len(train_df) + len(val_df)) * CFG['max_patches'] * (768 + 512) * 4 / 2**20
    print(f"Lazy CPU feature cache enabled (image tensors: {image_cache_mib:.1f} MiB, plus tabular data and overhead).")

eff_bs = min(CFG['batch_size'], max(4, len(train_df)))
train_dl = DataLoader(PanCancerDataset(train_df, rna_df, cli_df), batch_size=eff_bs, shuffle=True, generator=torch.Generator().manual_seed(CFG['seed']), num_workers=CFG['num_workers'])
val_dl   = DataLoader(PanCancerDataset(val_df, rna_df, cli_df), batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'])

model = CancerFusionResearchPlatform(rna_dim=rna_df.shape[1], cli_dim=cli_df.shape[1], embed_dim=64, max_slices=CFG['max_patches']).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
print(f"✅ Phase 2 Research Engine Ready on Device: {DEVICE}!")


In [ ]:
print("\n" + "="*65)
print("  🚀 IGNITING CANCERFUSION RESEARCH PLATFORM TRAINING ENGINE")
print("="*65)

history = {
    'epoch': [],
    'epoch_seconds': [],
    'total_loss': [],
    'cox_loss': [],
    'edl_loss': [],
    'orth_loss': [],
    'sparse_loss': []
}

from datetime import datetime, timezone
from time import perf_counter
run_dir = Path('/kaggle/working') / datetime.now(timezone.utc).strftime('phase2-%Y%m%d-%H%M%S-%f')
run_dir.mkdir()

model.train()
for epoch in range(1, CFG['epochs']+1):
    epoch_started = perf_counter()
    ep_total_loss = 0
    ep_cox_loss = 0
    ep_edl_loss = 0
    ep_orth_loss = 0
    ep_sparse_loss = 0
    
    successful_updates = 0
    pbar = tqdm(train_dl, desc=f'Epoch {epoch:02d}/{CFG["epochs"]}', leave=False)
    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)
        wsi = batch['wsi'].to(DEVICE)
        rad = batch['rad'].to(DEVICE)
        rna = batch['rna'].to(DEVICE)
        cli = batch['cli'].to(DEVICE)
        mask= batch['mask'].to(DEVICE)
        times = batch['time'].to(DEVICE)
        events= batch['event'].to(DEVICE)
        
        risk, unc, weights, xai, alpha = model(wsi, rad, rna, cli, mask, batch['wsi_valid'].to(DEVICE), batch['rad_valid'].to(DEVICE))
        loss, loss_dict = composite_research_loss(
            risk, times, events, alpha, xai['unimodal_tokens'], xai['rna_pathways'], modality_mask=mask
        )
        
        if not torch.isfinite(loss).item():
            raise ValueError(f"Non-finite training loss during epoch {epoch}. Review the inputs and training configuration.")
        loss.backward()
        optimizer.step()
        successful_updates += 1
        ep_total_loss += loss.item()
        ep_cox_loss += loss_dict['l_cox']
        ep_edl_loss += loss_dict['l_edl']
        ep_orth_loss += loss_dict['l_orth']
        ep_sparse_loss += loss_dict['l_sparse']

    if successful_updates == 0:
        raise ValueError("The training loader produced no batches.")
    num_batches = successful_updates
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize(DEVICE)
    elapsed = perf_counter() - epoch_started
    history['epoch'].append(epoch)
    history['epoch_seconds'].append(elapsed)
    history['total_loss'].append(ep_total_loss / num_batches)
    history['cox_loss'].append(ep_cox_loss / num_batches)
    history['edl_loss'].append(ep_edl_loss / num_batches)
    history['orth_loss'].append(ep_orth_loss / num_batches)
    history['sparse_loss'].append(ep_sparse_loss / num_batches)
    
    print(f"Epoch {epoch:02d}/{CFG['epochs']:02d} | Total Composite Loss: {ep_total_loss/num_batches:.4f} | Cox Hazard Loss: {ep_cox_loss/num_batches:.4f} | {elapsed:.1f} s")

torch.save(model.state_dict(), run_dir / 'CancerFusion_Phase2_Weights.pth')
pd.DataFrame(history).to_csv(run_dir / 'training_history.csv', index=False)
print(f"Training checkpoint and history saved to {run_dir}")


## Observed-outcome validation

Review the training and validation counts before interpreting the estimates. The paired ablations use cases with all four inputs available so that a comparison changes only the input mask. The optional clinical Cox baseline requires an explicit list of reviewed predictors.


In [ ]:
"""Evaluate the trained checkpoint once on observed, right-censored outcomes."""

import hashlib
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from lifelines.utils import concordance_index

output_dir = run_dir
figure_dir = output_dir / "paper_figures"
figure_dir.mkdir(parents=True, exist_ok=True)
modalities = ("wsi", "rad", "rna", "cli")


def observed_c_index(rows, risk_column):
    if rows.empty or not rows["event"].any():
        return None
    try:
        return float(concordance_index(rows["time"], -rows[risk_column], rows["event"]))
    except ZeroDivisionError:
        return None


def concordance_summary(rows, risk_columns, reference_column=None):
    """Use identical patient bootstrap draws for every score in a comparison."""
    rows = rows.reset_index(drop=True)
    points = {key: observed_c_index(rows, key) for key in risk_columns}
    samples = {key: [] for key in risk_columns}
    differences = {key: [] for key in risk_columns if reference_column and key != reference_column}
    rng = np.random.default_rng(CFG["seed"])
    if len(rows) and any(point is not None for point in points.values()):
        for _ in range(CFG["bootstrap_replicates"]):
            draw = rows.iloc[rng.integers(len(rows), size=len(rows))]
            scores = {key: observed_c_index(draw, key) for key in risk_columns}
            for key, score in scores.items():
                if score is not None:
                    samples[key].append(score)
                    if key in differences and scores[reference_column] is not None:
                        differences[key].append(scores[reference_column] - score)

    def interval(values):
        # Resamples without comparable pairs are visible in the reported count.
        if len(values) < 2:
            return None
        return np.percentile(values, [2.5, 97.5]).tolist()

    summary = {
        "patients": len(rows),
        "observed_events": int(rows["event"].sum()),
        "bootstrap_replicates_requested": CFG["bootstrap_replicates"],
        "scores": {},
    }
    for key, point in points.items():
        stats = {
            "harrell_c": point,
            "bootstrap_95_ci": interval(samples[key]),
            "bootstrap_replicates_valid": len(samples[key]),
        }
        if key in differences:
            reference = points[reference_column]
            stats.update(
                reference=reference_column,
                reference_minus_model_c=reference - point if point is not None and reference is not None else None,
                paired_difference_bootstrap_95_ci=interval(differences[key]),
                paired_difference_replicates_valid=len(differences[key]),
            )
        summary["scores"][key] = stats
    return summary


@torch.inference_mode()
def evaluate_validation(loader):
    model.eval()
    records = []
    for batch in tqdm(loader, desc="Validation inference"):
        tensors = [batch[key].to(DEVICE) for key in modalities]
        available = batch["mask"].to(DEVICE)
        if not (available > 0).any(dim=1).all().item():
            raise ValueError("A validation case has no model inputs. Review the modality audit.")
        risk, uncertainty, weights, _, _ = model(*tensors, available)
        if not all(torch.isfinite(value).all().item() for value in (risk, uncertainty, weights)):
            raise ValueError("Model produced non-finite validation outputs.")
        complete = (available > 0).all(dim=1)
        digest = {}
        if complete.any().item():
            complete_inputs = [value[complete] for value in tensors]
            for column, modality in enumerate(modalities):
                mask = available[complete].clone()
                mask[:, column] = 0
                masked_risk = model(*complete_inputs, mask)[0]
                if not torch.isfinite(masked_risk).all().item():
                    raise ValueError(f"Model produced non-finite scores with {modality} withheld.")
                result = torch.full_like(risk, float("nan"))
                result[complete] = masked_risk
                digest[f"without_{modality}_risk"] = result.cpu().numpy()
        frame = pd.DataFrame({
            "patient_id": batch["patient_id"],
            "time": batch["time"].cpu().numpy(),
            "event": batch["event"].cpu().numpy().astype(int),
            "fusion_risk": risk.cpu().numpy(),
            "evidence_uncertainty": uncertainty.cpu().numpy(),
        })
        for column, key in enumerate(modalities):
            frame[f"has_{key}"] = available[:, column].cpu().numpy().astype(bool)
            frame[f"weight_{key}"] = weights[:, column].cpu().numpy()
        for key, values in digest.items():
            frame[key] = values
        records.append(frame)
    predictions = pd.concat(records, ignore_index=True)
    for key in modalities:
        name = f"without_{key}_risk"
        if name not in predictions:
            predictions[name] = np.nan
    if predictions["patient_id"].duplicated().any():
        raise ValueError("The validation export contains duplicate patient identifiers.")
    return predictions


predictions = evaluate_validation(val_dl)
paired = predictions[predictions[[f"has_{key}" for key in modalities]].all(axis=1)].copy()
metrics = {
    "evaluation_scope": "Internal held-out validation, observed time-to-event and right censoring",
    "checkpoint_sha256": hashlib.sha256((output_dir / "CancerFusion_Phase2_Weights.pth").read_bytes()).hexdigest(),
    "seed": CFG["seed"],
    "training_patients": len(train_df),
    "training_observed_events": int(train_df["event"].sum()),
    "modality_counts": {key: int(predictions[f"has_{key}"].sum()) for key in modalities},
    "all_validation_cases": concordance_summary(predictions, ["fusion_risk"]),
    "paired_four_modality_cases": concordance_summary(
        paired, ["fusion_risk", *(f"without_{key}_risk" for key in modalities)], "fusion_risk"
    ),
}

# A baseline can only use explicitly reviewed predictors known at the prediction date.
# Features are selected and standardized using training rows alone.
approved = CFG["clinical_baseline_columns"]
predictions["clinical_cox_risk"] = np.nan
if not approved:
    metrics["clinical_cox_baseline"] = {"status": "not_run", "reason": "Configure reviewed clinical_baseline_columns."}
else:
    unknown = set(approved) - set(cli_df.columns)
    if unknown or len(approved) != len(set(approved)):
        raise ValueError(f"Clinical baseline columns are unknown or duplicated: {sorted(unknown)}")
    training = train_df.set_index("Patient_ID").join(cli_df[approved], how="inner")
    test = predictions[predictions["has_cli"]].set_index("patient_id").join(cli_df[approved], how="inner")
    varying = training[approved].std(ddof=0)
    used = varying[varying.gt(0)].index.tolist()
    if test.empty or not used or not training["event"].any():
        metrics["clinical_cox_baseline"] = {"status": "not_run", "reason": "Insufficient eligible clinical inputs or training events."}
    else:
        mean, scale = training[used].mean(), varying[used]
        clinical = CoxPHFitter(penalizer=0.1)
        try:
            clinical.fit(
                ((training[used] - mean) / scale).join(training[["time", "event"]]),
                duration_col="time", event_col="event",
            )
        except ConvergenceError as exc:
            metrics["clinical_cox_baseline"] = {"status": "failed_to_converge", "reason": str(exc)}
        else:
            risk = clinical.predict_log_partial_hazard((test[used] - mean) / scale)
            predictions["clinical_cox_risk"] = predictions["patient_id"].map(risk)
            matched = predictions.dropna(subset=["clinical_cox_risk"])
            metrics["clinical_cox_baseline"] = {
                "status": "evaluated",
                "training_patients": len(training),
                "training_events": int(training["event"].sum()),
                "columns": used,
                **concordance_summary(matched, ["fusion_risk", "clinical_cox_risk"], "fusion_risk"),
            }

predictions.to_csv(output_dir / "validation_predictions.csv", index=False, na_rep="")
(output_dir / "complete_metrics_summary.json").write_text(
    json.dumps(metrics, indent=2, allow_nan=False), encoding="utf-8"
)

fig, ax = plt.subplots(figsize=(8, 5))
for key, label in (("total_loss", "Total"), ("cox_loss", "Batch Cox"),
                   ("edl_loss", "Evidence penalty"), ("orth_loss", "Orthogonality")):
    ax.plot(history["epoch"], history[key], label=label)
ax.set(title="Training losses", xlabel="Epoch", ylabel="Loss")
ax.legend()
fig.tight_layout()
fig.savefig(figure_dir / "Figure1_Loss_Convergence.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
weights = predictions[[f"weight_{key}" for key in modalities]]
ax.bar([key.upper() for key in modalities], weights.mean(), yerr=weights.std(ddof=0), capsize=4)
ax.set(title="Validation pooling weights (mean and SD)", ylabel="Pooling weight", ylim=(0, 1))
fig.tight_layout()
fig.savefig(figure_dir / "Figure6_Modality_Importance.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(predictions["fusion_risk"], predictions["evidence_uncertainty"])
ax.set(title="Observed validation outputs", xlabel="Raw relative risk score", ylabel="Evidence uncertainty score")
fig.tight_layout()
fig.savefig(figure_dir / "Figure8_Risk_and_Evidence_Uncertainty.png", dpi=180)
plt.close(fig)

scores = metrics["paired_four_modality_cases"]["scores"]
names = [key for key, result in scores.items() if result["harrell_c"] is not None]
if names:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(names[::-1], [scores[key]["harrell_c"] for key in names[::-1]])
    ax.set(xlim=(0, 1), xlabel="Harrell C-index", title="Comparison on the same four-modality cases")
    fig.tight_layout()
    fig.savefig(figure_dir / "Figure11_Paired_Ablations.png", dpi=180)
    plt.close(fig)

print(json.dumps(metrics, indent=2, allow_nan=False))
if metrics["all_validation_cases"]["scores"]["fusion_risk"]["harrell_c"] is None:
    print("Validation contains no comparable observed-event pairs. Clinical discrimination is unavailable.")
print("Saved validation predictions, survival metrics and computed figures.")


In [ ]:
# Package the artifacts belonging to this training run.
import shutil

shutil.copytree(audit_dir, run_dir / 'data_audit', dirs_exist_ok=True)
archive = shutil.make_archive(str(run_dir), 'zip', run_dir)
print(f"Download {archive} from the Kaggle Output tab.")
print("Bundle includes the checkpoint, data audit, training history, validation predictions, metrics and figures.")
